# PyTestLab Notebook GUI

Build interactive controls for supported instrument operations with
`ipywidgets`. All three instruments below use local simulation fixtures.

Use Python 3.11 or newer and install PyTestLab and ipywidgets into the
environment of your Jupyter kernel: `python -m pip install pytestlab ipywidgets`.

In [ ]:
# 1. Create (simulated) instruments
from pytestlab import AutoInstrument


def main():
    # Create simulated instruments using current PyTestLab patterns
    psu = AutoInstrument.from_config("keysight/EDU36311A", simulate=True)

    dmm = AutoInstrument.from_config("keysight/34470A", simulate=True)

    awg = AutoInstrument.from_config("keysight/EDU33212A", simulate=True)

    print("Simulated PSU, DMM & AWG ready → build GUI panels below…")
    return psu, dmm, awg


# Create instruments
psu, dmm, awg = main()

In [ ]:
# 2. Build a PSU control panel (Channel 1)
from pytestlab.gui.builder import Button
from pytestlab.gui.builder import InstrumentPanel
from pytestlab.gui.builder import Slider
from pytestlab.gui.builder import Toggle

# Use chainable facade API as shown in README
psu_panel = InstrumentPanel(
    title="PSU – CH1 (Sim)",
    inst=psu,
    controls=[
        Slider(
            label="Voltage [V]",
            getter=lambda i: i.channel(1).get_voltage(),
            setter=lambda i, v: i.channel(1).set(voltage=v),
            min=0.0,
            max=6.0,
            step=0.05,
        ),
        Toggle(
            label="Output",
            getter=lambda i: i.get_output_state(1),
            setter=lambda i, state: i.channel(1).on() if state else i.channel(1).off(),
            on_desc="ON",
            off_desc="OFF",
        ),
        Button(
            label="⟳ Refresh read-back",
            action=lambda i, p: p.refresh(),
        ),
    ],
)

# Move the slider, toggle the output – the PSU simulation follows immediately.
psu_panel

In [ ]:
# 3. Build a DMM one-click read-out
dmm_panel = InstrumentPanel(
    title="DMM – DC Voltage (Sim)",
    inst=dmm,
    controls=[
        Button(
            label="Measure DC Voltage",
            action=lambda dev, p: print("Measured:", dev.measure("dc_voltage")),
            style="success",
        ),
        Button(
            label="Measure AC Voltage",
            action=lambda dev, p: print("Measured:", dev.measure("ac_voltage")),
            style="info",
        ),
        Button(
            label="Measure Current",
            action=lambda dev, p: print("Measured:", dev.measure("dc_current")),
            style="warning",
        ),
    ],
)

# Click **Measure** buttons – the values are printed using the current API
dmm_panel

In [ ]:
# 4. Build an AWG control panel using current patterns
awg_panel = InstrumentPanel(
    title="AWG – CH1 (Sim)",
    inst=awg,
    controls=[
        Slider(
            label="Frequency [Hz]",
            getter=lambda i: i.get_frequency(1),
            setter=lambda i, f: i.set_frequency(1, f),
            min=1.0,
            max=1000000.0,
            step=100.0,
        ),
        Slider(
            label="Amplitude [Vpp]",
            getter=lambda i: i.get_amplitude(1),
            setter=lambda i, a: i.set_amplitude(1, a),
            min=0.01,
            max=5.0,
            step=0.01,
        ),
        Toggle(
            label="Output",
            getter=lambda i: i.get_output_state(1),
            setter=lambda i, state: i.channel(1).enable() if state else i.channel(1).disable(),
            on_desc="ON",
            off_desc="OFF",
        ),
        Button(
            label="Generate Sine Wave",
            action=lambda i, p: i.channel(1).setup_sine(frequency=1000.0, amplitude=1.0),
            style="primary",
        ),
    ],
)

# Control AWG parameters with the modern chainable API
awg_panel

## Finish the demonstration

When you are done using the widgets, close the simulated instruments:

```python
for instrument in (awg, dmm, psu):
    instrument.close()
```

To adapt panels for a real bench, use supported profiles and separately verify
resource addresses, wiring, instrument limits and the behavior of each control.
This notebook does not open hardware.